# GFF Parsing with Biopython (BCBio)

This notebook demonstrates how to parse a GFF3 file using `bcbio-gff` as suggested in the [Biopython Wiki](https://biopython.org/wiki/GFF_Parsing).

In [1]:
from BCBio import GFF
from Bio import SeqIO
import os
import pandas as pd
import os
import glob
import subprocess
import shutil
import gc

# MOL_TYPE

In [2]:
# 1. Define Paths
data_dir = '../viral_data_all/ncbi_dataset/data_subset/'

# 2. Identify the target FASTA files
# finding all files starting with GCA, GCF, ending with _genomic.fna, without 'cds'
file_pattern = '**/genomic.gff'

gff_files = [f for f in glob.glob(os.path.join(data_dir, file_pattern), recursive=True) if 'cds' not in os.path.basename(f)]

print(f"Found {len(gff_files)} GFF files matching the pattern.")

Found 211307 GFF files matching the pattern.


In [3]:
# --- Function: extract mol_type and strand from a GFF file ---
def extract_mol_type(gff_file):
    """Extract mol_type and strand from region features in a GFF file.
    Returns a list of dicts with record_id, mol_type, strand, and accession."""
    
    # Extract accession folder name from path
    # e.g., "../viral_data_all/ncbi_dataset/data_subset/GCF_000864765.1/genomic.gff"
    #  → "GCF_000864765.1"
    accession = os.path.basename(os.path.dirname(gff_file))
    
    results = []
    try:
        with open(gff_file) as in_handle:
            for record in GFF.parse(in_handle):
                for feature in record.features:
                    if feature.type == "region":
                        mol_type = feature.qualifiers.get("mol_type", [None])[0]
                        # Extract strand: 1 -> '+', -1 -> '-', 0/None -> '.'
                        raw_strand = feature.location.strand
                        results.append({
                            "record_id": record.id,
                            "accession": accession,
                            "region_mol_type": mol_type,
                            "region_strand": raw_strand,
                            "source_file": gff_file
                        })
    except Exception as e:
        print(f"Error parsing {gff_file}: {e}")
        # Still try raw parsing as fallback
        try:
            results.extend(extract_mol_type_raw(gff_file))
        except Exception as e2:
            print(f"  Raw fallback also failed: {e2}")
    
    return results


    
def extract_mol_type_raw(gff_file):
    """Fallback: parse GFF as plain text to extract mol_type and strand from region lines."""
    accession = os.path.basename(os.path.dirname(gff_file))
    results = []
    with open(gff_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 9 and parts[2] == "region":
                attrs = parts[8]
                record_id = parts[0]
                # Strand is column 7 (index 6)
                strand = parts[6] if len(parts) > 6 else '.'
                mol_type = None
                for attr in attrs.split(";"):
                    if attr.startswith("mol_type="):
                        mol_type = attr.split("=", 1)[1]
                        break
                results.append({
                    "record_id": record_id,
                    "accession": accession,
                    "region_mol_type": mol_type,
                    "region_strand": strand,
                    "source_file": gff_file
                })
    return results

In [ ]:
all_matches = []
## Error files already handled within the original function ####
checkpoint_dir = "../gff_results/mol_type_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_interval = 10000
checkpoint_count = 0
for i, gff_file in enumerate(gff_files):
    matches = extract_mol_type(gff_file)
    all_matches.extend(matches)
    
    # Checkpoint: save and flush every 10,000 files
    if (i + 1) % checkpoint_interval == 0:
        if all_matches:
            df_chunk = pd.DataFrame(all_matches)
            df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
            print(f"[{i+1}/{len(gff_files)}] Saved checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
            checkpoint_count += 1
            all_matches = []  # free memory
            del df_chunk
            gc.collect()
        else:
            print(f"[{i+1}/{len(gff_files)}] No matches in this batch, skipping checkpoint.")
# Save any remaining matches after the loop
if all_matches:
    df_chunk = pd.DataFrame(all_matches)
    df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
    print(f"[{len(gff_files)}/{len(gff_files)}] Saved final checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
    del df_chunk
    gc.collect()
# # --- Merge all checkpoints into one DataFrame ---
# checkpoint_files = sorted(
#     [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
# )
# print(f"\nMerging {len(checkpoint_files)} checkpoint files...")
# df_mol = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
# df_mol = df_mol.drop_duplicates(subset=["record_id", "accession", "mol_type"])
# print(f"Found {len(df_mol)} unique records with mol_type across {len(gff_files)} files.")
# print(f"\nmol_type value counts:")
# print(df_mol["mol_type"].value_counts())
# display(df_mol.head(20))

In [ ]:
checkpoint_dir = "../gff_results/mol_type_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# --- Merge all checkpoints into one DataFrame ---
checkpoint_files = sorted(
    [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
)
print(f"\nMerging {len(checkpoint_files)} checkpoint files from {checkpoint_dir}...")

df_all = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["record_id", "accession", "region_mol_type", "source_file"])

df_all['region_mol_type']

## removes duplicates with NA values, keeping strand
df_all = df_all.groupby(['record_id', 'accession']).agg(
    region_mol_type=('region_mol_type', 'first'),
    region_strand=('region_strand', 'first')
).dropna(subset=['region_mol_type']).reset_index()

print(f"Found {len(df_all)} unique features.")
display(df_all.head(20))

print(f"\nstrand value counts:")
print(df_all["region_strand"].value_counts())

df_all.to_csv("../tables/mol_type_parsed.csv", index=False)
print("Saved combined df to mol_type_parsed.csv")


Merging 22 checkpoint files from ../gff_results/mol_type_checkpoints...
Found 1326715 unique features.


,record_id,accession,region_mol_type,region_strand
0,AB000403.1,GCA_000847645.1,genomic RNA,1
1,AB000404.1,GCA_000847645.1,genomic RNA,1
2,AB000851.1,GCA_000871205.1,genomic RNA,1
3,AB000923.1,GCA_000840045.1,genomic DNA,1
4,AB000924.1,GCA_000840045.1,genomic DNA,1
5,AB000925.1,GCA_000840045.1,genomic DNA,1
6,AB000926.1,GCA_000840045.1,genomic DNA,1
7,AB000927.1,GCA_000840045.1,genomic DNA,1
8,AB005795.1,GCA_000855625.1,genomic RNA,1
9,AB006783.1,GCA_000853245.1,genomic RNA,1



strand value counts:
region_strand
1    815745
1    510897
+        73
Name: count, dtype: int64
Saved combined df to mol_type_parsed.csv


In [19]:
df_all.region_strand.unique()

array([1, '1', '+'], dtype=object)

# CHECKING


In [ ]:
df = pd.read_csv("../tables/mol_type_parsed.csv")

In [ ]:
df.head()


df.groupby('accession')['mol_type'].nunique().loc[lambda x: x > 1]
df.groupby('accession')['mol_type']



In [37]:
from BCBio import GFF
gff_file = "../viral_data_all/ncbi_dataset/data_subset/GCA_000840725.1/genomic.gff"
with open(gff_file) as in_handle:
    for record in GFF.parse(in_handle):
        print(f"Record: {record.id}")
        for feature in record.features:
            print(f"  Feature: {feature.type}, {feature.location}")
            # To see mol_type if it exists in the top-level record:
            # print(record.annotations.get('molecule_type'))

Record: AY357582.2
  Feature: region, [0:57455](+)
  Feature: sequence_feature, [0:12](+)
  Feature: gene, [641:941](-)
  Feature: gene, [951:1383](-)
  Feature: gene, [1372:1990](-)
  Feature: gene, [2012:2339](-)
  Feature: gene, [2341:2785](-)
  Feature: gene, [2787:3780](-)
  Feature: gene, [3790:3937](-)
  Feature: gene, [4025:4352](-)
  Feature: gene, [4341:4767](-)
  Feature: gene, [4804:5287](-)
  Feature: gene, [5325:6210](-)
  Feature: gene, [6432:6720](+)
  Feature: gene, [6785:7067](+)
  Feature: gene, [7066:7297](+)
  Feature: gene, [7283:7514](+)
  Feature: gene, [7516:7765](+)
  Feature: gene, [7829:8513](+)
  Feature: gene, [8497:8806](+)
  Feature: gene, [8802:9096](+)
  Feature: gene, [9118:9685](+)
  Feature: gene, [9681:10395](+)
  Feature: gene, [10476:12636](+)
  Feature: gene, [12637:12910](+)
  Feature: gene, [12952:13420](+)
  Feature: gene, [13474:14017](+)
  Feature: gene, [14018:14519](+)
  Feature: gene, [14493:14898](+)
  Feature: gene, [14894:15173](+)
  

In [38]:
import pandas as pd
gff_file = "../viral_data_all/ncbi_dataset/data_subset/GCA_000820495.2/genomic.gff"
# GFF files are tab-separated; we skip lines starting with '#'
df_gff = pd.read_csv(
    gff_file, 
    sep='\t', 
    comment='#', 
    header=None,
    names=["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]
)
# To see the mol_type, it's usually inside the 'attributes' column for 'region' type features
df_gff

,seqid,source,type,start,end,score,strand,phase,attributes
0,M14880.1,Genbank,region,1,2368,.,+,.,ID=M14880.1:1..2368;Dbxref=taxon:518987;Name=1...
1,M14880.1,Genbank,CDS,21,2279,.,+,0,ID=cds-AAA43767.1;Dbxref=NCBI_GP:AAA43767.1;Na...
2,AF101982.1,Genbank,region,1,2313,.,+,.,ID=AF101982.1:1..2313;Dbxref=taxon:518987;gbke...
3,AF101982.1,Genbank,gene,1,2313,.,+,.,ID=gene-PB2;Name=PB2;gbkey=Gene;gene=PB2;gene_...
4,AF101982.1,Genbank,CDS,1,2313,.,+,0,ID=cds-AAF06851.1;Parent=gene-PB2;Dbxref=NCBI_...
5,AF102017.1,Genbank,region,1,2204,.,+,.,ID=AF102017.1:1..2204;Dbxref=taxon:518987;gbke...
6,AF102017.1,Genbank,gene,1,2181,.,+,.,ID=gene-PA;Name=PA;gbkey=Gene;gene=PA;gene_bio...
7,AF102017.1,Genbank,CDS,1,2181,.,+,0,ID=cds-AAF06886.1;Parent=gene-PA;Dbxref=NCBI_G...
8,K00423.1,Genbank,region,1,1882,.,+,.,ID=K00423.1:1..1882;Dbxref=taxon:518987;Name=4...
9,K00423.1,Genbank,CDS,34,1788,.,+,0,ID=cds-AAA43716.1;Dbxref=NCBI_GP:AAA43716.1;Na...


In [ ]:

cols_with_na = df.columns[df.isna().any()].tolist()
cols_with_na

rows_with_na = df[df.isna().any(axis=1)]
rows_with_na




In [ ]:
mol_type = pd.read_csv("../tables/mol_type_parsed.csv")
mol_type.region_strand = 1
mol_type.to_csv("../tables/mol_type_parsed.csv")